# Gold Layer - Price Band Performance View

## Purpose
Analyze product performance by price segments (budget, economy, standard, premium, luxury, ultra) to identify pricing sweet spots.

## Type
**SQL View** (lightweight, real-time)

## Input
* **Source:** `big_data.silver.order_products` (33.8M rows)
* **Source:** `big_data.silver.products_enriched` (49.7K rows)

## Output
* **Target:** `big_data.gold.vw_price_band_performance`
* **Rows:** 6 (price bands)
* **Refresh:** Real-time (always reflects current Silver data)

## Use Cases
* 💰 Pricing strategy optimization
* 🎯 Identify best-performing price segments
* 📈 Price elasticity analysis

## Why View (not Table)?
* ✅ Result is very small (6 rows)
* ✅ Query is fast (simple join + groupBy)
* ✅ Always synchronized with Silver
* ✅ Zero storage overhead

## SQL Logic
1. Join order_products with products_enriched (price_band)
2. GROUP BY price_band
3. COUNT orders and calculate reorder rate per band

## Execution
Run all cells sequentially. Expected runtime: ~25 seconds.

In [0]:
%sql
-- Price Band Performance View

CREATE OR REPLACE VIEW big_data.gold.vw_price_band_performance AS
SELECT 
  p.price_band,
  COUNT(*) AS times_ordered,
  ROUND(SUM(CASE WHEN op.reordered THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS reorder_rate
FROM big_data.silver.order_products op
LEFT JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
GROUP BY p.price_band
ORDER BY p.price_band;

In [0]:
%sql
-- Verify view exists and preview price bands
SELECT * FROM big_data.gold.vw_price_band_performance;